# ARC spatial-program induction with one global Qwen adapter

This notebook trains one fixed Qwen3 LoRA adapter on four L4 GPUs to infer executable spatial programs from ARC-style demonstrations.

The synthetic data changes the actual rule and scene on every episode. Rotation and recoloring appear as ordinary primitives; they are not used to turn one puzzle into hundreds of nominal samples.

Primary metric: parse the generated JSON program, execute it on a hidden query grid, and compare the resulting grid exactly. Select Kaggle's 4 x L4 accelerator and attach the private model `aishikai/qwen3-4b-instruct-2507-unsloth-4bit` plus the private dataset `aishikai/offline-unsloth-trl-wheelhouse-py311` before running.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

WHEELHOUSE = Path("/kaggle/input/offline-unsloth-trl-wheelhouse-py311")
requirements = WHEELHOUSE / "requirements.in"
assert requirements.exists(), "Attach the private dataset aishikai/offline-unsloth-trl-wheelhouse-py311"
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", "--find-links", str(WHEELHOUSE), "-r", str(requirements),
], check=True)
required = ["unsloth", "trl", "datasets", "tensorboard"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f"Offline dependency installation failed: {missing}"
print("Offline dependencies are available.")


In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 3407
SMOKE_TEST = False
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/models/aishikai/qwen3-4b-instruct-2507-unsloth-4bit/transformers/bnb-4bit/1"),
    Path("/kaggle/input/qwen3-4b-instruct-2507-unsloth-4bit/transformers/bnb-4bit/1"),
]
MODEL_PATH = next((str(path) for path in MODEL_PATH_CANDIDATES if (path / "config.json").exists()), None)
if MODEL_PATH is None and Path("/kaggle/input").exists():
    matches = list(Path("/kaggle/input").glob("**/qwen3-4b-instruct-2507-unsloth-4bit/**/config.json"))
    MODEL_PATH = str(matches[0].parent) if matches else None
assert MODEL_PATH, "Attach the private Kaggle Model aishikai/qwen3-4b-instruct-2507-unsloth-4bit"
MAX_SEQ_LENGTH = 4096
TRAIN_EPISODES = 256 if SMOKE_TEST else 12_000
EVAL_EPISODES = 64 if SMOKE_TEST else 512
EVAL_GENERATIONS = 24 if SMOKE_TEST else 128

random.seed(SEED)
np.random.seed(SEED)
print({"train": TRAIN_EPISODES, "eval": EVAL_EPISODES, "model_path": MODEL_PATH})


## Executable spatial DSL

Programs are JSON arrays. Operations execute from left to right. The executor is deliberately small: it contains exact geometry while the model must infer which operations and parameters explain the demonstrations.

In [ ]:
DIRECTIONS = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1),
}


def as_grid(grid):
    a = np.asarray(grid, dtype=int)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        raise ValueError("grid must be rectangular and at most 30x30")
    if a.min() < 0 or a.max() > 9:
        raise ValueError("colors must be 0..9")
    return a


def background(grid):
    a = as_grid(grid)
    counts = Counter(a.ravel().tolist())
    return min(counts, key=lambda color: (-counts[color], color))


def components(grid):
    a = as_grid(grid)
    bg = background(a)
    seen = set()
    out = []
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            color = int(a[r, c])
            if color == bg or (r, c) in seen:
                continue
            q = [(r, c)]
            seen.add((r, c))
            cells = []
            while q:
                rr, cc = q.pop()
                cells.append((rr, cc))
                for dr, dc in DIRECTIONS.values():
                    nr, nc = rr + dr, cc + dc
                    if (
                        0 <= nr < a.shape[0]
                        and 0 <= nc < a.shape[1]
                        and (nr, nc) not in seen
                        and int(a[nr, nc]) == color
                    ):
                        seen.add((nr, nc))
                        q.append((nr, nc))
            rs = [x[0] for x in cells]
            cs = [x[1] for x in cells]
            out.append({
                "cells": cells,
                "color": color,
                "size": len(cells),
                "r": min(rs),
                "c": min(cs),
                "h": max(rs) - min(rs) + 1,
                "w": max(cs) - min(cs) + 1,
            })
    return out


def select_object(grid, selector):
    objs = components(grid)
    if not objs:
        raise ValueError("no foreground objects")
    keys = {
        "largest": lambda o: (-o["size"], o["r"], o["c"]),
        "smallest": lambda o: (o["size"], o["r"], o["c"]),
        "topmost": lambda o: (o["r"], o["c"], -o["size"]),
        "bottommost": lambda o: (-(o["r"] + o["h"]), o["c"], -o["size"]),
        "leftmost": lambda o: (o["c"], o["r"], -o["size"]),
        "rightmost": lambda o: (-(o["c"] + o["w"]), o["r"], -o["size"]),
    }
    if selector not in keys:
        raise ValueError(f"unknown selector: {selector}")
    return sorted(objs, key=keys[selector])[0]


def crop_foreground(grid):
    a = as_grid(grid)
    bg = background(a)
    cells = np.argwhere(a != bg)
    if not len(cells):
        return a.tolist()
    r0, c0 = cells.min(axis=0)
    r1, c1 = cells.max(axis=0)
    return a[r0:r1 + 1, c0:c1 + 1].tolist()


def recolor_object(grid, selector, color):
    a = as_grid(grid).copy()
    obj = select_object(a, selector)
    for r, c in obj["cells"]:
        a[r, c] = int(color)
    return a.tolist()


def recolor_foreground(grid, color):
    a = as_grid(grid).copy()
    bg = background(a)
    a[a != bg] = int(color)
    return a.tolist()


def move_object(grid, selector, direction, steps):
    a = as_grid(grid).copy()
    bg = background(a)
    obj = select_object(a, selector)
    dr, dc = DIRECTIONS[direction]
    moved = [(r + dr * steps, c + dc * steps) for r, c in obj["cells"]]
    own = set(obj["cells"])
    for r, c in moved:
        if not (0 <= r < a.shape[0] and 0 <= c < a.shape[1]):
            raise ValueError("move leaves grid")
        if int(a[r, c]) != bg and (r, c) not in own:
            raise ValueError("move collides")
    for r, c in obj["cells"]:
        a[r, c] = bg
    for r, c in moved:
        a[r, c] = obj["color"]
    return a.tolist()


def extract_object(grid, selector):
    a = as_grid(grid)
    bg = background(a)
    obj = select_object(a, selector)
    out = np.full((obj["h"], obj["w"]), bg, dtype=int)
    for r, c in obj["cells"]:
        out[r - obj["r"], c - obj["c"]] = obj["color"]
    return out.tolist()


def complete_symmetry(grid, axis):
    a = as_grid(grid).copy()
    bg = background(a)
    source = a.copy()
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            if int(source[r, c]) == bg:
                continue
            rr, cc = (r, a.shape[1] - 1 - c) if axis == "vertical" else (a.shape[0] - 1 - r, c)
            if int(a[rr, cc]) == bg:
                a[rr, cc] = source[r, c]
    return a.tolist()


def connect_markers(grid):
    a = as_grid(grid).copy()
    singletons = [o for o in components(a) if o["size"] == 1]
    pairs = []
    for i, first in enumerate(singletons):
        for second in singletons[i + 1:]:
            if first["color"] != second["color"]:
                continue
            p = first["cells"][0]
            q = second["cells"][0]
            if p[0] == q[0] or p[1] == q[1]:
                pairs.append((p, q, first["color"]))
    if len(pairs) != 1:
        raise ValueError("expected one aligned equal-color marker pair")
    (r1, c1), (r2, c2), color = pairs[0]
    if r1 == r2:
        a[r1, min(c1, c2):max(c1, c2) + 1] = color
    else:
        a[min(r1, r2):max(r1, r2) + 1, c1] = color
    return a.tolist()


def trace_path(grid, orientation):
    a = as_grid(grid)
    bg = background(a)
    cells = {tuple(x) for x in np.argwhere(a != bg)}
    if not cells:
        raise ValueError("empty path")
    neighbors = {
        p: [
            (p[0] + dr, p[1] + dc)
            for dr, dc in DIRECTIONS.values()
            if (p[0] + dr, p[1] + dc) in cells
        ]
        for p in cells
    }
    endpoints = sorted(p for p, ns in neighbors.items() if len(ns) == 1)
    if len(endpoints) != 2 or any(len(ns) > 2 for ns in neighbors.values()):
        raise ValueError("foreground is not one simple path")
    order = []
    previous = None
    current = endpoints[0]
    while current is not None:
        order.append(current)
        nxt = [p for p in neighbors[current] if p != previous]
        previous, current = current, (nxt[0] if nxt else None)
    if len(order) != len(cells):
        raise ValueError("path is disconnected")
    values = [int(a[r, c]) for r, c in order]
    return ([values] if orientation == "row" else [[x] for x in values])


def copy_marker_color(grid, selector):
    a = as_grid(grid)
    target = select_object(a, selector)
    markers = [o for o in components(a) if o["size"] == 1 and o["cells"] != target["cells"]]
    if len(markers) != 1:
        raise ValueError("expected one singleton marker")
    return recolor_object(a, selector, markers[0]["color"])


def upscale(grid, factor):
    a = as_grid(grid)
    return np.repeat(np.repeat(a, factor, axis=0), factor, axis=1).tolist()


def apply_program(grid, program):
    out = as_grid(grid).tolist()
    for step in program:
        op = step["op"]
        if op == "rotate":
            out = np.rot90(as_grid(out), -int(step["k"])).tolist()
        elif op == "flip":
            out = (np.fliplr(as_grid(out)) if step["axis"] == "vertical" else np.flipud(as_grid(out))).tolist()
        elif op == "recolor_object":
            out = recolor_object(out, step["selector"], step["color"])
        elif op == "recolor_foreground":
            out = recolor_foreground(out, step["color"])
        elif op == "move_object":
            out = move_object(out, step["selector"], step["direction"], step["steps"])
        elif op == "extract_object":
            out = extract_object(out, step["selector"])
        elif op == "crop_foreground":
            out = crop_foreground(out)
        elif op == "complete_symmetry":
            out = complete_symmetry(out, step["axis"])
        elif op == "connect_markers":
            out = connect_markers(out)
        elif op == "trace_path":
            out = trace_path(out, step["orientation"])
        elif op == "copy_marker_color":
            out = copy_marker_color(out, step["selector"])
        elif op == "upscale":
            out = upscale(out, int(step["factor"]))
        else:
            raise ValueError(f"unknown operation: {op}")
        as_grid(out)
    return out


def canonical_program(program):
    return json.dumps(program, separators=(",", ":"))


## Procedural episode generator

Each episode samples a program first, then creates three distinct demonstration scenes and one hidden query scene governed by that same program. Evaluation uses a separate seed range and includes compositions absent from training.

In [ ]:
DSL_SPEC = """Programs are JSON arrays executed left to right.
Allowed operations:
{"op":"rotate","k":1|2|3}
{"op":"flip","axis":"horizontal"|"vertical"}
{"op":"recolor_object","selector":SELECTOR,"color":0..9}
{"op":"recolor_foreground","color":0..9}
{"op":"move_object","selector":SELECTOR,"direction":"up"|"down"|"left"|"right","steps":1|2}
{"op":"extract_object","selector":SELECTOR}
{"op":"crop_foreground"}
{"op":"complete_symmetry","axis":"horizontal"|"vertical"}
{"op":"connect_markers"}
{"op":"trace_path","orientation":"row"|"column"}
{"op":"copy_marker_color","selector":SELECTOR}
{"op":"upscale","factor":2|3}
SELECTOR is largest, smallest, topmost, bottommost, leftmost, or rightmost.
Colors are integers 0..9. Return only the JSON array."""

SYSTEM_PROMPT = (
    "Infer the shortest valid spatial program that explains every demonstration. "
    "Return only one JSON array. Do not return prose or the query grid."
)

TRAIN_FAMILIES = [
    "rotate", "flip", "recolor_object", "move_object", "extract_object",
    "crop", "upscale", "symmetry", "connect", "trace", "color_transfer",
    "rotate_recolor",
]
EVAL_FAMILIES = TRAIN_FAMILIES + ["flip_recolor", "crop_upscale"]


def literal_colors(program):
    return {int(step["color"]) for step in program if "color" in step}


def sample_program(family, rng):
    selectors = ["largest", "smallest", "topmost", "bottommost", "leftmost", "rightmost"]
    if family == "rotate":
        return [{"op": "rotate", "k": int(rng.integers(1, 4))}]
    if family == "flip":
        return [{"op": "flip", "axis": rng.choice(["horizontal", "vertical"]).item()}]
    if family == "recolor_object":
        return [{"op": "recolor_object", "selector": rng.choice(selectors).item(), "color": int(rng.integers(0, 10))}]
    if family == "move_object":
        return [{
            "op": "move_object",
            "selector": rng.choice(["largest", "smallest"]).item(),
            "direction": rng.choice(list(DIRECTIONS)).item(),
            "steps": int(rng.integers(1, 3)),
        }]
    if family == "extract_object":
        return [{"op": "extract_object", "selector": rng.choice(selectors).item()}]
    if family == "crop":
        return [{"op": "crop_foreground"}]
    if family == "upscale":
        return [{"op": "upscale", "factor": int(rng.integers(2, 4))}]
    if family == "symmetry":
        return [{"op": "complete_symmetry", "axis": rng.choice(["horizontal", "vertical"]).item()}]
    if family == "connect":
        return [{"op": "connect_markers"}]
    if family == "trace":
        return [{"op": "trace_path", "orientation": rng.choice(["row", "column"]).item()}]
    if family == "color_transfer":
        return [{"op": "copy_marker_color", "selector": "largest"}]
    if family == "rotate_recolor":
        return [
            {"op": "rotate", "k": int(rng.integers(1, 4))},
            {"op": "recolor_foreground", "color": int(rng.integers(0, 10))},
        ]
    if family == "flip_recolor":
        return [
            {"op": "flip", "axis": rng.choice(["horizontal", "vertical"]).item()},
            {"op": "recolor_foreground", "color": int(rng.integers(0, 10))},
        ]
    if family == "crop_upscale":
        return [{"op": "crop_foreground"}, {"op": "upscale", "factor": 2}]
    raise ValueError(family)


def empty_cell(grid, r, c, margin=0):
    h, w = grid.shape
    r0, r1 = max(0, r - margin), min(h, r + margin + 1)
    c0, c1 = max(0, c - margin), min(w, c + margin + 1)
    bg = background(grid)
    return np.all(grid[r0:r1, c0:c1] == bg)


def object_scene(rng, forbidden=(), add_marker=False):
    h, w = (int(x) for x in rng.integers(8, 13, size=2))
    allowed = [x for x in range(10) if x not in set(forbidden)]
    bg = int(rng.choice(allowed))
    grid = np.full((h, w), bg, dtype=int)
    colors = [x for x in allowed if x != bg]
    rng.shuffle(colors)
    shapes = [(1, 2), (2, 2), (2, 3)]
    rng.shuffle(shapes)
    for index, (oh, ow) in enumerate(shapes):
        placed = False
        for _ in range(200):
            r = int(rng.integers(0, h - oh + 1))
            c = int(rng.integers(0, w - ow + 1))
            r0, r1 = max(0, r - 1), min(h, r + oh + 1)
            c0, c1 = max(0, c - 1), min(w, c + ow + 1)
            if np.all(grid[r0:r1, c0:c1] == bg):
                grid[r:r + oh, c:c + ow] = colors[index]
                placed = True
                break
        if not placed:
            raise ValueError("could not place objects")
    if add_marker:
        for _ in range(200):
            r, c = (int(x) for x in rng.integers([0, 0], [h, w]))
            if empty_cell(grid, r, c, margin=1):
                grid[r, c] = colors[3]
                break
        else:
            raise ValueError("could not place marker")
    return grid.tolist()


def symmetry_scene(rng, axis, forbidden=()):
    h, w = (int(x) for x in rng.integers(7, 12, size=2))
    allowed = [x for x in range(10) if x not in set(forbidden)]
    bg = int(rng.choice(allowed))
    colors = [x for x in allowed if x != bg]
    grid = np.full((h, w), bg, dtype=int)
    count = int(rng.integers(4, 9))
    for _ in range(count):
        if axis == "vertical":
            r = int(rng.integers(0, h))
            c = int(rng.integers(0, max(1, w // 2)))
        else:
            r = int(rng.integers(0, max(1, h // 2)))
            c = int(rng.integers(0, w))
        grid[r, c] = int(rng.choice(colors))
    return grid.tolist()


def marker_scene(rng, forbidden=()):
    h, w = (int(x) for x in rng.integers(7, 12, size=2))
    allowed = [x for x in range(10) if x not in set(forbidden)]
    bg = int(rng.choice(allowed))
    colors = [x for x in allowed if x != bg]
    marker = int(rng.choice(colors))
    grid = np.full((h, w), bg, dtype=int)
    if bool(rng.integers(0, 2)):
        r = int(rng.integers(1, h - 1))
        c1 = int(rng.integers(0, w - 3))
        c2 = int(rng.integers(c1 + 3, w))
        grid[r, c1] = grid[r, c2] = marker
    else:
        c = int(rng.integers(1, w - 1))
        r1 = int(rng.integers(0, h - 3))
        r2 = int(rng.integers(r1 + 3, h))
        grid[r1, c] = grid[r2, c] = marker
    return grid.tolist()


def path_scene(rng, forbidden=()):
    h, w = (int(x) for x in rng.integers(8, 13, size=2))
    allowed = [x for x in range(10) if x not in set(forbidden)]
    bg = int(rng.choice(allowed))
    colors = [x for x in allowed if x != bg]
    grid = np.full((h, w), bg, dtype=int)
    r0 = int(rng.integers(1, h - 4))
    c0 = int(rng.integers(1, w - 4))
    vertical = int(rng.integers(2, min(6, h - r0)))
    horizontal = int(rng.integers(2, min(6, w - c0)))
    coords = [(r, c0) for r in range(r0, r0 + vertical)]
    coords += [(r0 + vertical - 1, c) for c in range(c0 + 1, c0 + horizontal)]
    run_colors = rng.choice(colors, size=3, replace=False).tolist()
    cuts = sorted(rng.choice(range(1, len(coords)), size=2, replace=False).tolist())
    for i, (r, c) in enumerate(coords):
        color_index = 0 if i < cuts[0] else (1 if i < cuts[1] else 2)
        grid[r, c] = run_colors[color_index]
    return grid.tolist()


def make_scene(family, program, rng):
    forbidden = literal_colors(program)
    if family == "symmetry":
        return symmetry_scene(rng, program[0]["axis"], forbidden)
    if family == "connect":
        return marker_scene(rng, forbidden)
    if family == "trace":
        return path_scene(rng, forbidden)
    if family == "color_transfer":
        return object_scene(rng, forbidden, add_marker=True)
    if family == "upscale":
        return object_scene(rng, forbidden)
    return object_scene(rng, forbidden)


def render_grid(grid):
    return "\n".join("".join(str(int(x)) for x in row) for row in grid)


def build_user_prompt(demos, query):
    parts = [DSL_SPEC, "--- DEMONSTRATIONS ---"]
    for i, (inp, out) in enumerate(demos, 1):
        parts.append(f"Demo {i} input ({len(inp)}x{len(inp[0])}):\n{render_grid(inp)}")
        parts.append(f"Demo {i} output ({len(out)}x{len(out[0])}):\n{render_grid(out)}")
    parts.append(f"--- QUERY ---\nInput ({len(query)}x{len(query[0])}):\n{render_grid(query)}")
    parts.append("Return the program only.")
    return "\n\n".join(parts)


def make_episode(seed, families):
    rng = np.random.default_rng(seed)
    family = rng.choice(families).item()
    program = sample_program(family, rng)
    examples = []
    attempts = 0
    while len(examples) < 4 and attempts < 500:
        attempts += 1
        try:
            inp = make_scene(family, program, rng)
            out = apply_program(inp, program)
            if out != inp and (inp, out) not in examples:
                examples.append((inp, out))
        except ValueError:
            continue
    if len(examples) != 4:
        raise RuntimeError(f"failed to generate {family} episode")
    demos = examples[:3]
    query_input, query_output = examples[3]
    user = build_user_prompt(demos, query_input)
    target = canonical_program(program)
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": target},
        ],
        "family": family,
        "program": target,
        "query_input": query_input,
        "query_output": query_output,
    }


def build_rows(count, start_seed, families):
    return [make_episode(start_seed + i, families) for i in range(count)]


In [ ]:
train_rows = build_rows(TRAIN_EPISODES, SEED, TRAIN_FAMILIES)
eval_rows = build_rows(EVAL_EPISODES, 10_000_000 + SEED, EVAL_FAMILIES)

for row in random.sample(train_rows, min(32, len(train_rows))):
    program = json.loads(row["program"])
    assert apply_program(row["query_input"], program) == row["query_output"]
    assert "QUERY OUTPUT" not in row["messages"][1]["content"]
    assert canonical_program(program) == row["program"]

assert {row["family"] for row in eval_rows} >= {"flip_recolor", "crop_upscale"}
train_dataset = Dataset.from_list([{"messages": row["messages"]} for row in train_rows])
Path("data").mkdir(exist_ok=True)
Path("data/model_path.txt").write_text(MODEL_PATH)
train_dataset.save_to_disk("data/train_messages")
print(train_dataset)
print(Counter(row["family"] for row in train_rows))
print("\nSample target:", train_rows[0]["program"])
print(train_rows[0]["messages"][1]["content"][:1200])


## Global QLoRA training

This trains one adapter across all generated rules with Distributed Data Parallel: four processes, one model replica per L4, and a distinct data shard per process. TensorBoard logs are written under outputs/logs.

In [ ]:
%%writefile train_ddp.py
import os
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import torch

SEED = 3407
MODEL_PATH = Path("data/model_path.txt").read_text().strip()
MAX_SEQ_LENGTH = 4096
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
if WORLD_SIZE != 4:
    raise RuntimeError(f"Expected four DDP processes, got {WORLD_SIZE}")
torch.cuda.set_device(LOCAL_RANK)

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_from_disk
from trl import SFTConfig, SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    local_files_only=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

dataset = load_from_disk("data/train_messages")
def format_messages(batch):
    return {"text": [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in batch["messages"]
    ]}
dataset = dataset.map(
    format_messages,
    batched=True,
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
    keep_in_memory=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        dataset_num_proc=1,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        warmup_ratio=0.03,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir="outputs/checkpoints",
        logging_dir="outputs/logs",
        report_to="tensorboard",
        save_strategy="steps",
        save_steps=250,
        save_total_limit=2,
        ddp_find_unused_parameters=False,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()
trainer.accelerator.wait_for_everyone()
if trainer.is_world_process_zero():
    trainer.save_model("outputs/adapters")
    tokenizer.save_pretrained("outputs/adapters")


In [ ]:
gpu_count = torch.cuda.device_count()
print(f"CUDA GPUs visible: {gpu_count}")
assert gpu_count == 4, f"Select the Kaggle 4 x L4 accelerator; found {gpu_count} GPU(s)"
!nvidia-smi -L
!torchrun --standalone --nproc_per_node=4 train_ddp.py


In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs


## Executable held-out evaluation

The exact-program metric is intentionally secondary. Two syntactically different programs can be behaviorally equivalent; execution on the hidden grid is what matters.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/adapters",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    local_files_only=True,
)
FastLanguageModel.for_inference(model)


def extract_program(text):
    start = text.find("[")
    if start < 0:
        raise ValueError("no JSON array")
    program, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(program, list):
        raise ValueError("program is not a list")
    return program


@torch.inference_mode()
def predict_program(row):
    prompt_messages = row["messages"][:-1]
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def evaluate(rows):
    records = []
    for index, row in enumerate(rows):
        raw = predict_program(row)
        valid = False
        correct = False
        predicted = None
        error = None
        try:
            predicted = extract_program(raw)
            produced = apply_program(row["query_input"], predicted)
            valid = True
            correct = produced == row["query_output"]
        except Exception as exc:
            error = repr(exc)
        records.append({
            "family": row["family"],
            "valid": valid,
            "correct": correct,
            "exact_program": (
                predicted == json.loads(row["program"]) if valid else False
            ),
            "target": row["program"],
            "prediction": raw,
            "error": error,
        })
        if (index + 1) % 10 == 0:
            print(f"{index + 1}/{len(rows)}")
    return records


eval_subset = eval_rows[:EVAL_GENERATIONS]
records = evaluate(eval_subset)

def rate(key, rows):
    return sum(bool(row[key]) for row in rows) / max(1, len(rows))

print({
    "episodes": len(records),
    "valid_program_rate": rate("valid", records),
    "exact_program_rate": rate("exact_program", records),
    "execution_accuracy": rate("correct", records),
})

for family in sorted({row["family"] for row in records}):
    subset = [row for row in records if row["family"] == family]
    print(f"{family:18s} n={len(subset):3d} execution={rate('correct', subset):.3f}")


In [ ]:
for record in records[:5]:
    print(json.dumps(record, indent=2)[:1800], "\n")

assert Path("outputs/adapters").exists()
print("Global ARC spatial-program adapter: outputs/adapters")


## Next experiment

Compare this global program model against the existing direct-grid Qwen solver on held-out official training tasks. Keep the two answer families separate so attempt 2 remains a genuinely different hypothesis.